In [6]:
# Install Databricks SQL Connector
!pip install databricks-sql-connector pandas -q

In [7]:
!pip install pyspark -q

In [ ]:
from google.colab import drive
import os

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
    print("✅ Drive mounted successfully")
else:
    print("✅ Drive already mounted, skipping...")

In [8]:
#configuration notebook

import os
os.chdir("/content/drive/My Drive/Colab Notebooks")
%run oo_config.ipynb

In [9]:
# ─── Databricks Connection Config ───────────────────────────────────────────
SERVER_HOSTNAME = G_SERVER_HOSTNAME
HTTP_PATH       = G_HTTP_PATH   # from Connection Details tab
ACCESS_TOKEN    = G_ACCESS_TOKEN         # replace with your PAT

# ─── Catalog / Schema / Table Target ────────────────────────────────────────
CATALOG   = "workspace"            # change to your catalog name (Unity Catalog)
SCHEMA    = "default"         # change to your schema/database
TABLE     = "test_colab_insert_spark"
FULL_TABLE = f"{CATALOG}.{SCHEMA}.{TABLE}"

In [10]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, BooleanType
# ── Session ───────────────────────────────────────────────────────────────────
spark = SparkSession.builder \
    .appName("MyEarthQuakeApp") \
    .getOrCreate()

print("Spark version:", spark.version)

schema = StructType([
    StructField("id",        IntegerType(), False),
    StructField("name",      StringType(),  False),
    StructField("score",     DoubleType(),  False),
    StructField("is_active", BooleanType(), False),
])

data = [
    (1, "Alice",   92.5, True),
    (2, "Bob",     87.0, False),
    (3, "Charlie", 78.3, True),
    (4, "Diana",   95.1, True),
    (5, "Eve",     88.9, False),
]

df = spark.createDataFrame(data, schema)
#── Example Spark transformations ──────────────────────────────────────────
df_processed = df \
    .filter(F.col("is_active") == True) \
    .withColumn("score_grade", F.when(F.col("score") >= 90, "A")
                                .when(F.col("score") >= 80, "B")
                                .otherwise("C")) \
    .withColumn("processed_at", F.current_timestamp())

df_processed.show(truncate=False)
df_processed.printSchema()

Spark version: 4.0.2
+---+-------+-----+---------+-----------+--------------------------+
|id |name   |score|is_active|score_grade|processed_at              |
+---+-------+-----+---------+-----------+--------------------------+
|1  |Alice  |92.5 |true     |A          |2026-02-27 08:23:55.831885|
|3  |Charlie|78.3 |true     |C          |2026-02-27 08:23:55.831885|
|4  |Diana  |95.1 |true     |A          |2026-02-27 08:23:55.831885|
+---+-------+-----+---------+-----------+--------------------------+

root
 |-- id: integer (nullable = false)
 |-- name: string (nullable = false)
 |-- score: double (nullable = false)
 |-- is_active: boolean (nullable = false)
 |-- score_grade: string (nullable = false)
 |-- processed_at: timestamp (nullable = false)



In [11]:
from databricks import sql

def run_query(cursor, query):
    """Helper to execute and optionally fetch results."""
    cursor.execute(query)
    try:
        return cursor.fetchall()
    except Exception:
        return None

with sql.connect(
    server_hostname = SERVER_HOSTNAME,
    http_path       = HTTP_PATH,
    access_token    = ACCESS_TOKEN
) as conn:
    with conn.cursor() as cursor:

        # ── Step 1: Set catalog and schema context ───────────────────────────
        cursor.execute(f"USE CATALOG {CATALOG}")
        cursor.execute(f"USE SCHEMA {SCHEMA}")

        # ── Step 2: Create table if not exists ───────────────────────────────
        create_sql = f"""
        CREATE TABLE IF NOT EXISTS {FULL_TABLE} (
            id        INT,
            name      STRING,
            score     DOUBLE,
            is_active BOOLEAN
        )
        """
        cursor.execute(create_sql)
        print(f"✅ Table '{FULL_TABLE}' created/verified.")

        # ── Step 3: Insert rows from Pandas DataFrame ─────────────────────────
        #for _, row in pdf.iterrows():
        #    insert_sql = f"""
        #    INSERT INTO {FULL_TABLE} (id, name, score, is_active)
        #    VALUES ({row['id']}, '{row['name']}', {row['score']}, {str(row['is_active']).upper()})
        #    """
        #    cursor.execute(insert_sql)

        # Spark equivalent of: [tuple(r) for r in df.itertuples(index=False)]
        rows = [tuple(row) for row in df.collect()]

        cursor.executemany(
            f"INSERT INTO {FULL_TABLE} VALUES (?, ?, ?, ?)",
            rows
        )

        print(f"✅ Inserted {df.count()} rows into '{FULL_TABLE}'.")

        # ── Step 4: Verify by reading back ───────────────────────────────────
        results = run_query(cursor, f"SELECT * FROM {FULL_TABLE}")
        print(f"\n📦 Data in '{FULL_TABLE}':")
        for r in results:
            print(r)

✅ Table 'workspace.default.test_colab_insert_spark' created/verified.
✅ Inserted 5 rows into 'workspace.default.test_colab_insert_spark'.

📦 Data in 'workspace.default.test_colab_insert_spark':
Row(id=3, name='Charlie', score=78.30000305175781, is_active=True)
Row(id=3, name='Charlie', score=78.30000305175781, is_active=True)
Row(id=3, name='Charlie', score=78.30000305175781, is_active=True)
Row(id=4, name='Diana', score=95.0999984741211, is_active=True)
Row(id=4, name='Diana', score=95.0999984741211, is_active=True)
Row(id=4, name='Diana', score=95.0999984741211, is_active=True)
Row(id=1, name='Alice', score=92.5, is_active=True)
Row(id=1, name='Alice', score=92.5, is_active=True)
Row(id=1, name='Alice', score=92.5, is_active=True)
Row(id=5, name='Eve', score=88.9000015258789, is_active=False)
Row(id=5, name='Eve', score=88.9000015258789, is_active=False)
Row(id=5, name='Eve', score=88.9000015258789, is_active=False)
Row(id=2, name='Bob', score=87.0, is_active=False)
Row(id=2, name='B

In [14]:
%%script echo "Skipping..."

from google.colab import drive
drive.mount('/content/drive')

import nbformat
from nbconvert.preprocessors import ExecutePreprocessor

notebook_path = "/content/drive/My Drive/Colab Notebooks/oo_config.ipynb"

with open(notebook_path) as f:
    nb = nbformat.read(f, as_version=4)

ep = ExecutePreprocessor(timeout=600, kernel_name="python3")
ep.preprocess(nb)

print("✅ Notebook executed successfully")


Skipping...
